# GitLab Catalog Discovery

GitLab Catalog Discovery durchsucht eine GitLab-Gruppe regelmässig nach `catalog-info.yaml`-Dateien und übernimmt die gefundenen Entities automatisch in den Backstage Software Catalog.

Das Modul ergänzt den Backstage Catalog um den GitLab Entity Provider, welcher GitLab-Projekte durchsucht und deren Catalog-Dateien einliest.

In [ ]:
%%bash
source ~/.nvm/nvm.sh
cd ~/mybackstage/
yarn --cwd packages/backend add @backstage/plugin-catalog-backend-module-gitlab

## Backend-Modul registrieren

Das Catalog-Modul wird im Backstage-Backend registriert, damit der konfigurierte GitLab Provider beim Start geladen wird.

In [ ]:
%%bash
cd ~/mybackstage/
sed -i "/backend.start();/i \\
backend.add(import('@backstage/plugin-catalog-backend-module-gitlab'));" \
  packages/backend/src/index.ts


## GitLab Integration konfigurieren

Die Integration definiert die GitLab-Instanz und den Zugriffstoken, über welchen Backstage die Projekte und Repository-Inhalte lesen kann.

In [ ]:
%%bash
cd ~/mybackstage/
cat >> app-config.local.yaml <<'EOF'

integrations:
  gitlab:
    - host: gitlab.com
      token: ${GITLAB_TOKEN}
EOF


## GitLab Discovery Provider konfigurieren

Der Provider durchsucht die angegebene GitLab-Gruppe alle 30 Minuten nach `catalog-info.yaml`-Dateien und ignoriert archivierte sowie geforkte Projekte.

In [ ]:
%%bash
source ~/.nvm/nvm.sh
cd ~/mybackstage/
cat >> app-config.local.yaml <<'EOF'

catalog:
  providers:
    gitlab:
      production:
        host: gitlab.com
        group: DEINE-GRUPPE
        entityFilename: catalog-info.yaml
        skipForkedRepos: true
        includeArchivedRepos: false
        schedule:
          frequency:
            minutes: 30
          timeout:
            minutes: 3
EOF


## Selbst betriebene GitLab-Instanz

Bei einer selbst betriebenen GitLab-Instanz müssen `host` und bei Bedarf `apiBaseUrl` angepasst werden.

In [ ]:
%%bash
source ~/.nvm/nvm.sh
cd ~/mybackstage/
cat <<'EOF'
integrations:
  gitlab:
    - host: gitlab.example.ch
      apiBaseUrl: https://gitlab.example.ch/api/v4
      token: ${GITLAB_TOKEN}

catalog:
  providers:
    gitlab:
      production:
        host: gitlab.example.ch
        group: platform
        entityFilename: catalog-info.yaml
        schedule:
          frequency:
            minutes: 30
          timeout:
            minutes: 3
EOF


## Catalog-Datei im GitLab-Projekt

Jedes zu importierende Repository benötigt die konfigurierte Datei, normalerweise `catalog-info.yaml`, auf dem Standard-Branch.

In [ ]:
%%bash
source ~/.nvm/nvm.sh
cd ~/mybackstage/
cat <<'EOF'
apiVersion: backstage.io/v1alpha1
kind: Component
metadata:
  name: example-service
  description: Beispielservice aus GitLab Catalog Discovery
spec:
  type: service
  lifecycle: production
  owner: user:default/guest
EOF
